<a href="https://colab.research.google.com/github/liangjingyi1211/neural-search-ranking/blob/main/notebooks/02_dense_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Dense Retrieval — BGE Small + FAISS (FiQA Dev)

**目标**：不训练模型，使用预训练 `BAAI/bge-small-en-v1.5` 将全部 FiQA 文档编码为 384 维向量，通过 FAISS 精确向量检索，对 **dev** 划分做与 BM25 相同的检索评测。

- 实验设置：FiQA dev；文档库全部 57,638 篇；Top-100；向量 L2 归一化 + 内积（等价于余弦相似度）。
- GPU 负责文本编码，FAISS CPU 精确检索；不使用 test 标签调参。
- 结果：`reports/fiqa_bge_small_dev.json`、`reports/fiqa_bge_small_dev_top100.json`；可选 Drive 缓存向量索引。

> 按从上到下顺序执行；Colab 运行时重置后，从头执行。不要把数据集、模型权重或大索引提交到 GitHub。

## 0. 初始化 Colab、克隆项目及检查 GPU

In [1]:
from pathlib import Path
import os, sys, time, json, random, hashlib
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path('/content/neural-search-ranking')
REPO_URL = 'https://github.com/liangjingyi1211/neural-search-ranking.git'
if not (PROJECT_DIR / '.git').exists():
    !git clone {REPO_URL}
os.chdir(PROJECT_DIR)
(PROJECT_DIR / 'reports').mkdir(parents=True, exist_ok=True)
print('Working directory:', os.getcwd())
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


Cloning into 'neural-search-ranking'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 27 (delta 3), reused 20 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 5.40 MiB | 11.51 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Working directory: /content/neural-search-ranking
Python: 3.13.16 PyTorch: 2.14.1+cu130
GPU: Tesla T4


## 1. 安装依赖
首次运行或 Colab 重置后执行。这里**不**重新安装 PyTorch。

In [2]:
%pip install -q sentence-transformers faiss-cpu beir pandas matplotlib

In [3]:
import faiss
import sentence_transformers
from sentence_transformers import SentenceTransformer
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
print('sentence-transformers:', sentence_transformers.__version__)
print('faiss:', faiss.__version__)


sentence-transformers: 5.7.0
faiss: 1.15.1


## 2. 加载 FiQA 的 **dev** 查询和完整文档库
`GenericDataLoader` 每个划分用新的对象；当前 Notebook 不需要加载 train，也绝不使用 test 标注。

In [4]:
DATASET = 'fiqa'
url = 'https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip'
data_path = util.download_and_unzip(url, str(PROJECT_DIR / 'data'))
corpus, dev_queries, dev_qrels = GenericDataLoader(data_folder=data_path).load(split='dev')

doc_ids = list(corpus.keys())
doc_texts = [((corpus[d].get('title') or '') + ' ' + (corpus[d].get('text') or '')).strip() for d in doc_ids]
query_ids = list(dev_queries.keys())
query_texts = [dev_queries[q] for q in query_ids]

assert len(doc_ids) == len(doc_texts) == len(set(doc_ids))
assert len(query_ids) == len(query_texts) == len(set(query_ids))
assert all(d in corpus for rels in dev_qrels.values() for d in rels)
assert all(q in dev_queries for q in dev_qrels)
print('Documents:', len(doc_ids), 'Dev queries:', len(query_ids), 'Qrels queries:', len(dev_qrels))
print('Document sample:', repr(doc_texts[0][:160]))
print('Query sample:', repr(query_texts[0]))


/content/neural-search-ranking/data/fiqa.zip:   0%|          | 0.00/17.1M [00:00<?, ?iB/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

Documents: 57638 Dev queries: 500 Qrels queries: 500
Document sample: "I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're buildi"
Query sample: 'Claiming business expenses for a business with no income'


## 3. 加载预训练 BGE Small
**Bi-Encoder 原理**：Query 和 Document 经同一编码器分别映射成 384 维向量；`normalize_embeddings=True` 对每行向量做 L2 归一化。此阶段不更新权重。

In [5]:
MODEL_NAME = 'BAAI/bge-small-en-v1.5'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 64 if DEVICE == 'cuda' else 16
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.eval()
print('Model:', MODEL_NAME, 'Device:', DEVICE)
print('Embedding dimension:', model.get_sentence_embedding_dimension())

# 先用两个句子测试张量形状
probe = model.encode(['stock market', 'portfolio investment'], normalize_embeddings=True,
                     convert_to_numpy=True)
print('Probe shape:', probe.shape, 'dtype:', probe.dtype)
assert probe.shape == (2, 384)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: BAAI/bge-small-en-v1.5 Device: cuda
Embedding dimension: 384


/tmp/ipykernel_1222/1554704950.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('Embedding dimension:', model.get_sentence_embedding_dimension())


Probe shape: (2, 384) dtype: float32


## 4. 可选：连接 Google Drive 保存/复用 FAISS 索引
Drive 缓存会减少下次重新编码整个语料的开销。文件约数十到一百 MB。若不想授权 Drive，把 `USE_DRIVE_CACHE` 设置为 `False`。

In [6]:
USE_DRIVE_CACHE = True
CACHE_DIR = None
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path('/content/drive/MyDrive/neural-search-ranking/artifacts/fiqa_bge_small_v1_5')
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ARTIFACT_DIR = PROJECT_DIR / 'data' / 'artifacts' / 'fiqa_bge_small_v1_5'
LOCAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 与 doc_ids 排序/语料内容绑定，避免载入不匹配的旧索引
h = hashlib.sha256()
for did, txt in zip(doc_ids, doc_texts):
    h.update(did.encode('utf-8')); h.update(b'\0')
    h.update(txt.encode('utf-8')); h.update(b'\0')
corpus_hash = h.hexdigest()
print('Corpus fingerprint:', corpus_hash[:16])


Mounted at /content/drive
Corpus fingerprint: acdd1b2139a0a388


## 5. 批量编码文档（或读取索引缓存）
输入：`doc_texts` 为 `[N]` 字符串；输出：`[N,384]` 的 `float32` 向量。

索引使用 `faiss.IndexFlatIP(384)`：**精确内积搜索，不是近似 ANN**。对已归一化向量，内积即余弦相似度。

In [7]:
import shutil
INDEX_NAME = 'fiqa_bge_small_flatip.faiss'
META_NAME = 'fiqa_bge_small_flatip_meta.json'
index_path = LOCAL_ARTIFACT_DIR / INDEX_NAME
meta_path = LOCAL_ARTIFACT_DIR / META_NAME

loaded_from_cache = False
index = None
encoding_seconds = None

def valid_metadata(meta):
    return (meta.get('corpus_sha256') == corpus_hash and
            meta.get('model_name') == MODEL_NAME and
            meta.get('document_count') == len(doc_ids) and
            meta.get('embedding_dim') == 384 and
            meta.get('normalized') is True)

if CACHE_DIR is not None and (CACHE_DIR / INDEX_NAME).is_file() and (CACHE_DIR / META_NAME).is_file():
    candidate_meta = json.loads((CACHE_DIR / META_NAME).read_text())
    if valid_metadata(candidate_meta):
        shutil.copy2(CACHE_DIR / INDEX_NAME, index_path)
        shutil.copy2(CACHE_DIR / META_NAME, meta_path)
        index = faiss.read_index(str(index_path))
        if index.ntotal == len(doc_ids) and index.d == 384:
            loaded_from_cache = True
            print('Loaded FAISS index from Drive cache.')
        else:
            index = None

if not loaded_from_cache:
    start = time.perf_counter()
    doc_embeddings = model.encode(
        doc_texts, batch_size=BATCH_SIZE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    )
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    encoding_seconds = time.perf_counter() - start
    doc_embeddings = np.ascontiguousarray(doc_embeddings, dtype=np.float32)
    assert doc_embeddings.shape == (len(doc_ids), 384)
    print('Document embeddings:', doc_embeddings.shape, doc_embeddings.dtype)

    index = faiss.IndexFlatIP(doc_embeddings.shape[1])
    index.add(doc_embeddings)
    faiss.write_index(index, str(index_path))
    meta = {
        'model_name': MODEL_NAME, 'corpus_sha256': corpus_hash,
        'document_count': len(doc_ids), 'embedding_dim': 384,
        'normalized': True, 'faiss_index': 'IndexFlatIP',
        'doc_ids': doc_ids, 'document_encoding_seconds': encoding_seconds
    }
    meta_path.write_text(json.dumps(meta, ensure_ascii=False), encoding='utf-8')
    if CACHE_DIR is not None:
        shutil.copy2(index_path, CACHE_DIR / INDEX_NAME)
        shutil.copy2(meta_path, CACHE_DIR / META_NAME)
    del doc_embeddings

print('Indexed docs:', index.ntotal, 'Dimension:', index.d)
print('Index size (MB):', round(index_path.stat().st_size / 1024**2, 2))
print('Document encoding seconds (new run only):', encoding_seconds)


Batches:   0%|          | 0/901 [00:00<?, ?it/s]

Document embeddings: (57638, 384) float32
Indexed docs: 57638 Dimension: 384
Index size (MB): 84.43
Document encoding seconds (new run only): 318.80950762899994


## 6. 编码 500 个 Dev 查询并搜索 Top-100
为了记录较可比的延迟，分别测量**整批 Query 编码**与**整批 FAISS 查询**时间；还会单独测逐条 Query 的端到端延迟（包含编码和搜索）。GPU 计时前后同步以避免异步计时偏差。

In [8]:
TOP_K = 100
start = time.perf_counter()
query_embeddings = model.encode(
    query_texts, batch_size=BATCH_SIZE, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True
)
if DEVICE == 'cuda':
    torch.cuda.synchronize()
batch_query_encoding_seconds = time.perf_counter() - start
query_embeddings = np.ascontiguousarray(query_embeddings, dtype=np.float32)
assert query_embeddings.shape == (len(query_ids), 384)

start = time.perf_counter()
faiss_scores, faiss_indices = index.search(query_embeddings, TOP_K)
batch_faiss_search_seconds = time.perf_counter() - start

# 按 doc_ids 中的原始顺序映射回文档 ID；每个查询得到一份 Top-100 结果
results = {}
for qi, qid in enumerate(query_ids):
    results[qid] = {
        doc_ids[int(di)]: float(score)
        for di, score in zip(faiss_indices[qi], faiss_scores[qi])
        if di >= 0
    }
assert len(results) == len(query_ids)
print('Query embedding shape:', query_embeddings.shape)
print('Batch query encode (s):', round(batch_query_encoding_seconds, 3))
print('Batch FAISS search (s):', round(batch_faiss_search_seconds, 3))


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Query embedding shape: (500, 384)
Batch query encode (s): 0.261
Batch FAISS search (s): 0.255


## 7. 使用 BEIR 官方接口计算检索指标
与 BM25 采用同样的 `dev_qrels`、Top-100、nDCG/Recall/MRR 评测协议。

In [9]:
k_values = [10, 100]
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    dev_qrels, results, k_values=k_values
)
mrr = EvaluateRetrieval.evaluate_custom(
    dev_qrels, results, k_values=[10], metric='mrr'
)
metrics = {
    'ndcg_10': ndcg['NDCG@10'],
    'recall_10': recall['Recall@10'],
    'recall_100': recall['Recall@100'],
    'mrr_10': mrr['MRR@10'],
}
print(json.dumps(metrics, indent=2))


{
  "ndcg_10": 0.39916,
  "recall_10": 0.46559,
  "recall_100": 0.70287,
  "mrr_10": 0.47566
}


## 8. 记录逐查询延迟（可选但建议）
逐条测量比批处理更接近 API 单查询的延迟。GPU/CPU 性能可能随共享运行时负载变化。为避免测量中的首次模型初始化影响结果，先进行 warm-up。

In [10]:
WARMUP = query_texts[0]
for _ in range(3):
    warm = model.encode([WARMUP], normalize_embeddings=True, convert_to_numpy=True)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    index.search(np.ascontiguousarray(warm, dtype=np.float32), TOP_K)

latencies_ms = []
for query in query_texts:
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    vec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    index.search(np.ascontiguousarray(vec, dtype=np.float32), TOP_K)
    latencies_ms.append((time.perf_counter() - start) * 1000)

print('End-to-end single-query P50 (ms):', round(float(np.percentile(latencies_ms, 50)), 2))
print('End-to-end single-query P95 (ms):', round(float(np.percentile(latencies_ms, 95)), 2))


End-to-end single-query P50 (ms): 19.36
End-to-end single-query P95 (ms): 32.48


## 9. 检查一个查询的 Top-5 文档
**相似度分数不是相关概率**；观察排名内容与 qrels 中的已标注相关文档是否重合。

In [11]:
qid = query_ids[0]
print('QUERY:', dev_queries[qid])
for rank, (did, score) in enumerate(sorted(results[qid].items(), key=lambda t: t[1], reverse=True)[:5], 1):
    label = dev_qrels.get(qid, {}).get(did, 0)
    print(f'\nRank {rank} | cosine={score:.4f} | qrel={label} | id={did}')
    print(doc_texts[doc_ids.index(did)][:350])


QUERY: Claiming business expenses for a business with no income

Rank 1 | cosine=0.8432 | qrel=1 | id=14255
Yes you can claim your business deductions if you are not making any income yet. But first you should decide what structure you want to have for your business. Either a Company structure or a Sole Trader or Partnership. Company Structure If you choose a Company Structure (which is more expensive to set up) you would claim your deductions but no inc

Rank 2 | cosine=0.7971 | qrel=0 | id=541809
"No, your business cannot deduct your non-business expenses.  You can only deduct from your business income those reasonable expenses you paid in order to earn income for the business. Moreover, for there to be a tax benefit, your business generally has to have income (but I expect there are exceptions; HST input tax credits come to mind.) The empl

Rank 3 | cosine=0.7950 | qrel=0 | id=529528
According to TurboTax, it is perfectly legal to final as an individual with zero income. But, alas, 

## 10. 生成报告，并与之前 BM25 Baseline 对照
报告自动保存到 `reports/`；**不要把大索引提交到 GitHub**。如果 BM25 JSON 在仓库中，会自动读取进行对比。

In [12]:
import pandas as pd
report = {
    'dataset': 'fiqa', 'split': 'dev', 'method': 'Dense_BGE_Small_FAISS_FlatIP',
    'model': MODEL_NAME, 'top_k': TOP_K,
    'corpus_size': len(doc_ids), 'query_count': len(query_ids),
    'embedding_dim': 384, 'normalized_embeddings': True,
    'faiss_index_type': 'IndexFlatIP', 'corpus_sha256': corpus_hash,
    **metrics,
    'index_size_mb': round(index_path.stat().st_size / 1024**2, 3),
    'document_encoding_seconds_this_run': encoding_seconds,
    'index_loaded_from_drive_cache': loaded_from_cache,
    'query_batch_encoding_seconds': round(batch_query_encoding_seconds, 4),
    'faiss_batch_search_seconds': round(batch_faiss_search_seconds, 4),
    'single_query_p50_ms': round(float(np.percentile(latencies_ms, 50)), 3),
    'single_query_p95_ms': round(float(np.percentile(latencies_ms, 95)), 3),
    'python_version': sys.version.split()[0],
    'torch_version': torch.__version__,
    'sentence_transformers_version': sentence_transformers.__version__,
    'faiss_version': faiss.__version__,
    'device': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
}
report_path = PROJECT_DIR / 'reports' / 'fiqa_bge_small_dev.json'
run_path = PROJECT_DIR / 'reports' / 'fiqa_bge_small_dev_top100.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
run_path.write_text(json.dumps(results, ensure_ascii=False), encoding='utf-8')

bm25_path = PROJECT_DIR / 'reports' / 'fiqa_bm25_dev.json'
rows = []
if bm25_path.exists():
    bm25 = json.loads(bm25_path.read_text(encoding='utf-8'))
    rows.append({'method': 'BM25', **{key: bm25.get(key) for key in metrics}})
else:
    print('BM25 report not found in repo; showing your earlier reported metrics for display only.')
    rows.append({'method': 'BM25 (previously reported)', 'ndcg_10': 0.22877,
                 'recall_10': 0.27244, 'recall_100': 0.47021, 'mrr_10': 0.28737})
rows.append({'method': 'BGE-Small + FAISS', **metrics})
print(pd.DataFrame(rows).to_string(index=False))
print('Saved:', report_path)
print('Saved:', run_path)

if CACHE_DIR is not None:
    shutil.copy2(report_path, CACHE_DIR / report_path.name)
    shutil.copy2(run_path, CACHE_DIR / run_path.name)
    print('Reports also backed up to:', CACHE_DIR)


BM25 report not found in repo; showing your earlier reported metrics for display only.
                    method  ndcg_10  recall_10  recall_100  mrr_10
BM25 (previously reported)  0.22877    0.27244     0.47021 0.28737
         BGE-Small + FAISS  0.39916    0.46559     0.70287 0.47566
Saved: /content/neural-search-ranking/reports/fiqa_bge_small_dev.json
Saved: /content/neural-search-ranking/reports/fiqa_bge_small_dev_top100.json
Reports also backed up to: /content/drive/MyDrive/neural-search-ranking/artifacts/fiqa_bge_small_v1_5


## 11. 保存与下一步

1. Colab 菜单 **File → Save a copy in GitHub**，路径 `notebooks/02_dense_retrieval.ipynb`。
2. 在 Terminal 中 `cd /content/neural-search-ranking && git status`，确认 `reports/fiqa_bge_small_dev.json` 与 `reports/fiqa_bge_small_dev_top100.json` 可被提交。若 `.gitignore` 过滤掉整个 `reports/`，需更改规则。
3. 提交报告之前检查输出是否包含隐私信息；**不要推送 `data/`、`.env`、Drive 缓存或模型权重**。
> Notebook 仅执行预训练推理，不会产生训练曲线；真正的对比学习训练将在 `03_embedding_training.ipynb` 中进行。